# TaxaLens v0.9 — Multi-source PoC + Genus Key Finder
Run one cell at a time. Large data stays on Google Drive; GitHub stores code only. Start with `doctor`, then download only the missing sources.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Keep the already completed Foundation_v06 data exactly where it is.
DATA_ROOT = '/content/drive/MyDrive/EntoKey/Foundation_v06'
BIOSCAN_ROOT = DATA_ROOT + '/raw/bioscan'
BIOSCAN_MANIFEST = DATA_ROOT + '/manifests/bioscan_raw.parquet'
REPO = '/content/TaxaLens'

In [ ]:
import os, subprocess
if not os.path.exists(REPO):
    subprocess.run(['git','clone','https://github.com/SaniyaSani/TaxaLens.git',REPO], check=True)
else:
    subprocess.run(['git','-C',REPO,'pull','--ff-only'], check=True)
os.chdir(REPO)
!pip -q install -r requirements-corpus.txt
!pip -q install torch torchvision transformers


## 1. Safe status check — downloads nothing

In [ ]:
!python scripts/run_multisource_poc_v09.py --stage doctor --data-root "{DATA_ROOT}" --bioscan-root "{BIOSCAN_ROOT}" --bioscan-manifest "{BIOSCAN_MANIFEST}"

## 2. Targeted BIOSCAN top-up
The first command counts the existing Muscidae and Tachinidae and downloads only the missing target-family JPEGs. The completed 30k selection and all existing images remain untouched. The normalized manifest is rebuilt in place.

In [ ]:
import pandas as pd
TARGET_FAMILIES = ['Muscidae', 'Tachinidae']
before = pd.read_parquet(BIOSCAN_MANIFEST)
print('Before top-up:')
display(before[before['family'].isin(TARGET_FAMILIES)].groupby(['family', 'source_split']).size().unstack(fill_value=0))
!python scripts/run_multisource_poc_v09.py --stage bioscan-topup --data-root "{DATA_ROOT}" --bioscan-root "{BIOSCAN_ROOT}" --bioscan-manifest "{BIOSCAN_MANIFEST}"
after = pd.read_parquet(BIOSCAN_MANIFEST)
print('After top-up:')
display(after[after['family'].isin(TARGET_FAMILIES)].groupby(['family', 'source_split']).size().unstack(fill_value=0))

## 3. Other sources, then rebuild the master corpus and strict 100k plan

In [ ]:
# Download only the other missing sources; BIOSCAN is reused here.
# !python scripts/run_multisource_poc_v09.py --stage download --data-root "{DATA_ROOT}" --bioscan-root "{BIOSCAN_ROOT}" --bioscan-manifest "{BIOSCAN_MANIFEST}" --reuse-existing-bioscan
!python scripts/run_multisource_poc_v09.py --stage ingest --data-root "{DATA_ROOT}" --bioscan-root "{BIOSCAN_ROOT}" --bioscan-manifest "{BIOSCAN_MANIFEST}"
!python scripts/run_multisource_poc_v09.py --stage assemble --data-root "{DATA_ROOT}" --bioscan-root "{BIOSCAN_ROOT}" --bioscan-manifest "{BIOSCAN_MANIFEST}"
!python scripts/run_multisource_poc_v09.py --stage plan --data-root "{DATA_ROOT}" --bioscan-root "{BIOSCAN_ROOT}" --bioscan-manifest "{BIOSCAN_MANIFEST}"

## 4. Cache images, then embed only one shard as a smoke test

In [ ]:
!python scripts/run_multisource_poc_v09.py --stage cache --data-root "{DATA_ROOT}" --bioscan-root "{BIOSCAN_ROOT}"
!python scripts/run_multisource_poc_v09.py --stage embed --data-root "{DATA_ROOT}" --bioscan-root "{BIOSCAN_ROOT}" --max-shards 1

## 5. Full embedding, training and measured report

In [ ]:
# Run only after the one-shard smoke test completed successfully.
# !python scripts/run_multisource_poc_v09.py --stage embed --data-root "{DATA_ROOT}" --bioscan-root "{BIOSCAN_ROOT}"
# !python scripts/run_multisource_poc_v09.py --stage train --data-root "{DATA_ROOT}" --bioscan-root "{BIOSCAN_ROOT}"
# !python scripts/run_multisource_poc_v09.py --stage evaluate --data-root "{DATA_ROOT}" --bioscan-root "{BIOSCAN_ROOT}"

## 6. Genus Key Finder smoke test

In [ ]:
!python scripts/find_genus_keys.py --family Syrphidae --genera Eristalis,Helophilus --offline